**CI twin of `ch06-backpropagation.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
import math

def sig(z):
    return 1 / (1 + math.exp(-z))

x = [1.0, 0.5]
target = 1.0
w1, b1 = [0.5, -0.5], 0.0     # hidden neuron 1
w2, b2 = [1.0, 1.0], -1.0     # hidden neuron 2
w3, b3 = [1.0, -1.0], 0.5     # output neuron

z1 = w1[0] * x[0] + w1[1] * x[1] + b1;  h1 = sig(z1)
z2 = w2[0] * x[0] + w2[1] * x[1] + b2;  h2 = sig(z2)
z3 = w3[0] * h1 + w3[1] * h2 + b3;      out = sig(z3)
loss = (out - target) ** 2

print(f"receipts: z1={z1:.4f}  h1={h1:.4f}   z2={z2:.4f}  h2={h2:.4f}")
print(f"          z3={z3:.4f}  out={out:.4f}   loss={loss:.4f}")

In [ ]:
dL_dout = 2 * (out - target)
print(f"dL/dout = 2·(0.6082 − 1) = {dL_dout:.4f}")

In [ ]:
dout_dz3 = out * (1 - out)            # sigmoid_slope, from the receipt
dL_dz3 = dL_dout * dout_dz3
print(f"dout/dz3 = {dout_dz3:.4f}   ->   dL/dz3 = {dL_dz3:.4f}")

In [ ]:
dL_dw3 = [dL_dz3 * h1, dL_dz3 * h2]
dL_db3 = dL_dz3
print(f"dL/dw3 = ({dL_dw3[0]:.4f}, {dL_dw3[1]:.4f})   dL/db3 = {dL_db3:.4f}")

In [ ]:
dL_dh1 = dL_dz3 * w3[0]
dL_dh2 = dL_dz3 * w3[1]
print(f"dL/dh1 = {dL_dh1:.4f}   dL/dh2 = {dL_dh2:.4f}")

In [ ]:
dL_dz1 = dL_dh1 * h1 * (1 - h1)
dL_dz2 = dL_dh2 * h2 * (1 - h2)

dL_dw1 = [dL_dz1 * x[0], dL_dz1 * x[1]];  dL_db1 = dL_dz1
dL_dw2 = [dL_dz2 * x[0], dL_dz2 * x[1]];  dL_db2 = dL_dz2

print(f"dL/dz1 = {dL_dz1:.5f}   dL/dz2 = {dL_dz2:.5f}")
print(f"dL/dw1 = ({dL_dw1[0]:.5f}, {dL_dw1[1]:.5f})")
print(f"dL/dw2 = ({dL_dw2[0]:.5f}, {dL_dw2[1]:.5f})")

In [ ]:
dL_dx1 = dL_dz1 * w1[0] + dL_dz2 * w2[0]
print(f"via neuron 1: {dL_dz1 * w1[0]:+.5f}")
print(f"via neuron 2: {dL_dz2 * w2[0]:+.5f}")
print(f"dL/dx1 (sum) = {dL_dx1:+.5f}")

In [ ]:
from lib.grader import run_tests, grad_check

def loss_of(params):
    a, b, c, d, e, f, g, h, i = params
    hh1 = sig(a * x[0] + b * x[1] + c)
    hh2 = sig(d * x[0] + e * x[1] + f)
    return (sig(g * hh1 + h * hh2 + i) - target) ** 2

params = [w1[0], w1[1], b1, w2[0], w2[1], b2, w3[0], w3[1], b3]
analytic = [dL_dw1[0], dL_dw1[1], dL_db1,
            dL_dw2[0], dL_dw2[1], dL_db2,
            dL_dw3[0], dL_dw3[1], dL_db3]

run_tests([
    grad_check("all nine hand-derived blames", loss_of, params, analytic),
])

In [ ]:
import time
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

digits = load_digits()
Xtr, Xte, ytr, yte = train_test_split(
    digits.data, digits.target, test_size=0.25,
    random_state=42, stratify=digits.target)
net = MLPClassifier(hidden_layer_sizes=(16,), activation="relu",
                    random_state=0, max_iter=2000).fit(Xtr, ytr)
W1n, W2n = net.coefs_
b1n, b2n = net.intercepts_
batch = Xte[:32]

def fwd():
    return np.maximum(0, batch @ W1n + b1n) @ W2n + b2n

t0 = time.perf_counter()
for _ in range(1211):                   # finite differences: n+1 passes
    fwd()
t_fd = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(2):                      # backprop: ~forward + backward
    fwd()
t_bp = time.perf_counter() - t0

print(f"finite differences: ~{t_fd * 1000:5.0f} ms per training step")
print(f"backprop:           ~{t_bp * 1000:5.2f} ms per training step  "
      f"(~{t_fd / t_bp:.0f}× cheaper)")

In [ ]:
z = np.array([5.2, 8.4, -12.4, -8.1, 5.4, -1.9, 2.2, -7.9, 6.8, -2.8])
truth = 1

e = np.exp(z - z.max())
p = e / e.sum()
onehot = np.zeros(10); onehot[truth] = 1.0

def ce_of_logits(zv):
    ev = np.exp(zv - zv.max())
    return -math.log((ev / ev.sum())[truth])

numeric = np.zeros(10)
for k in range(10):
    zp, zm = z.copy(), z.copy()
    zp[k] += 1e-6; zm[k] -= 1e-6
    numeric[k] = (ce_of_logits(zp) - ce_of_logits(zm)) / 2e-6

print("p − onehot:", np.round(p - onehot, 4))
print("numeric   :", np.round(numeric, 4))
print("agree:", np.allclose(p - onehot, numeric, atol=1e-5))

In [ ]:
dL_dout = 2 * (out - target)
dL_dz3 = dL_dout * out * (1 - out)
dL_dw3_0 = dL_dz3 * h1

run_tests([
    ("blame after the squash", round(dL_dz3, 4), -0.1867),
    ("the first weight's blame", round(dL_dw3_0, 4), -0.105),
])

In [ ]:
def sig(z):
    return 1 / (1 + math.exp(-z))

def backward_neuron(x, w, b, target):
    out = sig(sum(xi * wi for xi, wi in zip(x, w)) + b)
    dz = 2 * (out - target) * out * (1 - out)
    return [dz * xi for xi in x], dz

dw_a, db_a = backward_neuron([1.0, 2.0], [0.5, -0.25], 0.0, 1.0)
dw_b, db_b = backward_neuron([2.0, 0.0], [1.0, 1.0], -1.0, 0.0)

def loss_wrt_w(wv):
    o = sig(wv[0] * 1.0 + wv[1] * 2.0 + 0.0)
    return (o - 1.0) ** 2

run_tests([
    ("z=0 neuron: blame is clean", [round(v, 4) for v in dw_a] + [round(db_a, 4)],
     [-0.25, -0.5, -0.25]),
    ("second fixture", [round(v, 4) for v in dw_b] + [round(db_b, 4)],
     [0.5749, 0.0, 0.2875]),
    grad_check("the wiggle-test jury", loss_wrt_w, [0.5, -0.25], dw_a),
])